## Install Dependencies

In [ ]:
# Install dependencies
%pip install -q amplpy matplotlib

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["gurobi", "xpress", "highs", "gokestrel"], # modules to install
    license_uuid="default") # license to use

### AMPL Formulation
#### Parameter Bounds Are Enforced at Data Load Time

In [19]:
%%ampl_eval
reset;

set PROD default {'bands', 'coils', 'plate'};   # Set of products (with fallback default)

# Parameters indexed over PROD
param min_cost {p in PROD} default 10;          # Minimum allowed cost per product
param max_price {p in PROD} default 100;        # Maximum allowed price per product

# Parameters with bounds depending on p
param cost  {p in PROD} >= min_cost[p];         # Cost must be at least min_cost[p]
param price {p in PROD} in [cost[p], max_price[p]];  # Price must be between cost[p] and max_price[p]

# Load all parameter values in one table
data;
param: min_cost  max_price   cost    price :=
    bands 15    80          18      75
    coils 20    95          22      90    
    plate 25    120         30      110;

display cost, price;                            # Show final assigned values

:     cost price    :=
bands   18    75
coils   22    90
plate   30   110
;



#### ✅ What this does
Defines bounds that depend on other parameters
Loads data in a compact table format
Automatically checks that all values satisfy constraints
#### ❌ Invalid Data Example

In [ ]:
%%ampl_eval
reset data;
data;
param: min_cost  max_price   cost    price :=
    bands 15    80          18      75
    coils 20    95          22      90    
    plate 25    120         30      130;

display cost, price;  # [ERROR]: ❌ failed check: param price['plate'] = 130  

#### What happens?
For `plate`, the allowed range is: [30, 120] (`cost` = 30, `max_price` = 120). But `price` = 130 → ❌ violates constraint.
Parameter bounds (like `in [cost[p], max_price[p]]`) **are checked immediately when data is loaded**.
The model will not proceed until all data satisfies constraints.

#### Semantically equivalent declaration
Sometimes you don’t need to reference the index explicitly:

In [21]:
%%ampl_eval
reset;
set PROD default {'bands', 'coils', 'plate'};   # Declare a set with a default value

# Anonymous form — the attribute does not reference the iterated member
param cost  {PROD} >= 0;    # same bound applies to every member; p not needed
param price {PROD} >= 0;
param units {PROD} integer >= 1;

# Load all parameter values in one table
data;
param:   cost    price :=
    bands 18      75
    coils 22      90    
    plate 30      110;

display cost, price;                            # Show final assigned values

:     cost price    :=
bands   18    75
coils   22    90
plate   30   110
;



### amplpy Formulation
#### Define Model in AMPL

In [23]:
%%ampl_eval
reset;

set PROD default {'bands', 'coils', 'plate'};   # Set of products (with fallback default)

# Parameters indexed over PROD
param min_cost {p in PROD} default 10;          # Minimum allowed cost per product
param max_price {p in PROD} default 100;        # Maximum allowed price per product

# Parameters with bounds depending on p
param cost  {p in PROD} >= min_cost[p];         # Cost must be at least min_cost[p]
param price {p in PROD} in [cost[p], max_price[p]];  # Price must be between cost[p] and max_price[p]

#### Load Data Using pandas

In [25]:
import pandas as pd

# Create a table of parameter values indexed by products
df = pd.DataFrame({             # ‼️Column names must match parameter names    
    "min_cost": [15, 20, 25],
    "max_price": [80, 95, 120],
    "cost": [18, 22, 30],
    "price": [75, 90, 110]
}, index=["bands", "coils", "plate"])  # ‼️index must match set PROD

# Load each column into the corresponding AMPL parameter
for col in df.columns:
    ampl.param[col].set_values(df[col])  # assign values by product

# Verify loaded data
for param_name in df.columns:  # reuse df.columns instead of redefining
    print(f"{param_name}:\n{ampl.param[param_name].getValues()}\n")

min_cost:
   index0    |   min_cost  
  'bands'    |      15     
  'coils'    |      20     
  'plate'    |      25     


max_price:
   index0    |  max_price  
  'bands'    |      80     
  'coils'    |      95     
  'plate'    |     120     


cost:
   index0    |     cost    
  'bands'    |      18     
  'coils'    |      22     
  'plate'    |      30     


price:
   index0    |    price    
  'bands'    |      75     
  'coils'    |      90     
  'plate'    |     110     


